# SAE relabel pipeline — gooder corpus + 2026 captions + new tokenizer (Colab, A100)

Side pipeline for the **mega golf model**. One GPU pass over a MIXED corpus (web/code/CoT/math/chat),
harvest per-token GemmaScope firings, mine each latent's top firing spans with token witnesses, then a
cheap 2026 model on OpenRouter WRITES a fresh caption per latent and a second pass VALIDATES it
(caption must predict which spans fire). No old Neuronpedia labels needed anywhere.

Outputs one zip: `labels_final.json` (new captions) · `spans.jsonl.gz` + `targets.npz` (the training set,
per-token teacher firings) · `mega_tok.model` (32k BPE trained on this corpus) · `results.json`.

Cost (verified 2026-07-02, writer+validator, ~2.6k in + 190 out tok/latent): deepseek-v4-flash ≈ **$0.55 / 2k latents**
($1.35 / 5k) · gpt-5-mini ≈ $2.10 / 2k · haiku-4.5 ≈ $7 / 2k. GPU: one A100 session (harvest is the slow part).

In [2]:
# 1 · install
!pip -q install transformer-lens sae-lens scikit-learn "pandas<2.3" datasets huggingface_hub sentencepiece
!pip -q uninstall -y torchcodec torchvision torchaudio

In [3]:
# 2 · config — teacher, corpus mix, relabel models, quality gates
TEACHER = "9b"                       # "9b" = gemma-scope-9b layer 20 (new+useful) | "2b" = layer 12 (matches current app probe)
_T = {"9b": ("gemma-2-9b", "gemma-scope-9b-pt-res-canonical", "layer_20/width_16k/canonical"),
      "2b": ("gemma-2-2b", "gemma-scope-2b-pt-res-canonical", "layer_12/width_16k/canonical")}
MODEL_NAME, SAE_RELEASE, SAE_ID = _T[TEACHER]
DEVICE, DTYPE_NAME, RANDOM_SEED = "cuda", "bfloat16", 0

MIX = {"web": 1200, "code": 900, "cot": 900, "math": 600, "chat": 600}   # docs per domain (~8 spans/doc)
SPAN_MIN_T, SPAN_MAX_T, MAX_SPANS_PER_DOC = 12, 80, 12
ACT_FLOOR = 1.0                      # teacher activation that counts as a firing

DROP_TOP_MEAN, MIN_POS, MAX_FREQ, N_KEEP = 0.015, 30, 0.30, 2048         # latent keep filters (on the MIXED corpus)

WRITE_MODEL = "deepseek/deepseek-v4-flash"    # caption writer  (swap to "openai/gpt-5-mini" for ~4x cost)
CHECK_MODEL = "deepseek/deepseek-v4-flash"    # caption validator
N_SHOW, N_CONTRAST, N_VAL = 20, 8, 16         # spans per packet / validation quiz size
KEEP_SPECIFICITY, KEEP_VAL_F1 = 2, 0.60       # final gate: specific enough AND caption predicts firings
N_THREADS, TOK_VOCAB = 8, 32768

import os
OPENROUTER_KEY = os.environ.get("OPENROUTER_KEY", "")     # or paste: OPENROUTER_KEY = os.environ.get("OPENROUTER_KEY","")  # set your key in the environment
OUT = "/content/relabel_out"; os.makedirs(OUT, exist_ok=True)
print("config ready |", MODEL_NAME, SAE_ID, "| writer", WRITE_MODEL)

config ready | gemma-2-9b layer_20/width_16k/canonical | writer deepseek/deepseek-v4-flash


In [4]:
# 3 · HF login (gated Gemma; the-stack-smol auto-accepts on login)
from huggingface_hub import login
login()

In [5]:
# 4 · imports
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json, re, gc, random, math, time, heapq, gzip, threading
import numpy as np, torch
import concurrent.futures as cf
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
print("imports ready · device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

imports ready · device NVIDIA A100-SXM4-80GB


In [7]:
# 5 · teacher = Gemma + GemmaScope SAE (captions get written FROM firings — no old label file needed)
from transformer_lens import HookedTransformer
from sae_lens import SAE
DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16}[DTYPE_NAME]
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE, dtype=DTYPE)
_res = SAE.from_pretrained(SAE_RELEASE, SAE_ID, device=DEVICE)
sae = (_res[0] if isinstance(_res, tuple) else _res).to(DTYPE)
_meta = getattr(sae.cfg, "metadata", None)                    # sae-lens v6 moved hook info into cfg.metadata
HOOK = (getattr(sae.cfg, "hook_name", None) or getattr(_meta, "hook_name", None)
        or "blocks." + SAE_ID.split("/")[0].split("_")[1] + ".hook_resid_post")
D_SAE = getattr(sae.cfg, "d_sae", None) or sae.W_dec.shape[0]
PAD_ID = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
print("teacher loaded |", HOOK, "| d_sae", D_SAE)

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-9b into HookedTransformer
teacher loaded | blocks.20.hook_resid_post | d_sae 16384


In [8]:
# 6 · the GOODER corpus — stream a mix (web/code/cot/math/chat), sentence spans, domain-tagged
from datasets import load_dataset
WS_RE, SENT_RE = re.compile(r"[ \t]+"), re.compile(r"(?<=[.!?])\s+|\n{2,}|(?<=\n)(?=\S)")
def norm(t): return WS_RE.sub(" ", str(t).replace("\r", "")).strip()
def pick_text(rec):
    for k in ("text", "content", "solution", "response", "output", "chosen"):
        v = rec.get(k)
        if isinstance(v, str) and len(v) > 300: return v
    c = rec.get("conversations") or rec.get("messages")
    if isinstance(c, list):
        return "\n".join(str(m.get("value") or m.get("content") or "") for m in c if isinstance(m, dict))
    return ""
SOURCES = {  # domain -> fallback list of (dataset, config, split); all verified live + ungated (stack-smol = auto)
  "web":  [("HuggingFaceFW/fineweb", "sample-10BT", "train")],
  "code": [("bigcode/the-stack-smol", None, "train"), ("codeparrot/codeparrot-clean-valid", None, "train")],
  "cot":  [("open-thoughts/OpenThoughts-114k", None, "train")],
  "math": [("open-r1/OpenR1-Math-220k", None, "train")],
  "chat": [("Anthropic/hh-rlhf", None, "train")],
}
docs = []                                                     # (domain, text)
for dom, want in MIX.items():
    got = 0
    for name, dcfg, split in SOURCES[dom]:
        try: ds = load_dataset(name, dcfg, split=split, streaming=True)
        except Exception as e: print("  skip", name, "->", type(e).__name__); continue
        for rec in ds:
            t = norm(pick_text(rec))
            if len(t) > 400: docs.append((dom, t[:20000])); got += 1
            if got >= want: break
        if got: break
    print(f"  {dom}: {got} docs")
print("docs", len(docs))

spans = []                                                    # {i, doc, dom, ids, toks, text}
for di, (dom, text) in enumerate(docs):
    buf, n_from_doc = [], 0
    for sent in SENT_RE.split(text):
        sent = sent.strip()
        if sent: buf.append(sent)
        cand = " ".join(buf)
        ids = model.tokenizer(cand)["input_ids"]
        if ids and ids[0] == model.tokenizer.bos_token_id: ids = ids[1:]
        if len(ids) >= SPAN_MIN_T:
            ids = ids[:SPAN_MAX_T]
            toks = [model.tokenizer.convert_ids_to_tokens(i).replace("\u2581", " ").strip() or "·" for i in ids]
            spans.append({"i": len(spans), "doc": di, "dom": dom, "ids": ids, "toks": toks, "text": cand})
            buf, n_from_doc = [], n_from_doc + 1
            if n_from_doc >= MAX_SPANS_PER_DOC: break
    if di % 500 == 0: print(f"  docs {di} spans {len(spans)}")
from collections import Counter
print("spans", len(spans), "| by domain", dict(Counter(s["dom"] for s in spans)))

README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

  web: 1200 docs


README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

  skip bigcode/the-stack-smol -> DatasetNotFoundError


...

README.md:   0%|          | 0.00/5.13k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

  math: 600 docs


README.md:   0%|          | 0.00/5.77k [00:00<?, ?B/s]

  chat: 600 docs
docs 4200
  docs 0 spans 7
  docs 500 spans 5180
  docs 1000 spans 10226
  docs 1500 spans 15582
  docs 2000 spans 21130
  docs 2500 spans 27039
  docs 3000 spans 33039
  docs 3500 spans 37788
  docs 4000 spans 42010
spans 43669 | by domain {'web': 12285, 'code': 9942, 'cot': 10800, 'math': 5744, 'chat': 4898}


In [9]:
# 7 · ONE GPU pass — per-token SAE firings, sparse, over the whole mix (batched, RAM-resident)
gc.collect(); torch.cuda.empty_cache()
PS, PT, PL, PA = [], [], [], []                               # span, tok, latent, act — the sparse firing table
SPAN_HITS = np.zeros(D_SAE, np.int64); ACT_SUM = np.zeros(D_SAE, np.float64)
TOPS = [[] for _ in range(D_SAE)]                             # per-latent heap of (max_act_in_span, span_i)
order = sorted(range(len(spans)), key=lambda i: len(spans[i]["ids"]))
BS, total_tok, t0 = 24, 0, time.time()
with torch.no_grad():
    for b0 in range(0, len(order), BS):
        idxs = order[b0:b0 + BS]; L = max(len(spans[i]["ids"]) for i in idxs)
        ids = torch.full((len(idxs), L), PAD_ID, dtype=torch.long)
        for r, i in enumerate(idxs): ids[r, :len(spans[i]["ids"])] = torch.tensor(spans[i]["ids"])
        _, cache = model.run_with_cache(ids.to(DEVICE), names_filter=HOOK, return_type=None)
        feats = sae.encode(cache[HOOK]).float()               # [B, L, d_sae]; causal model -> right-pad is safe
        del cache
        for r, i in enumerate(idxs):
            n = len(spans[i]["ids"]); total_tok += n
            f = feats[r, :n]
            tt, ll = (f > ACT_FLOOR).nonzero(as_tuple=True)
            aa = f[tt, ll].cpu().numpy(); tt = tt.cpu().numpy().astype(np.int16); ll = ll.cpu().numpy().astype(np.int32)
            PS.append(np.full(len(ll), i, np.int32)); PT.append(tt); PL.append(ll); PA.append(aa.astype(np.float16))
            np.add.at(ACT_SUM, ll, aa.astype(np.float64))
            o2 = np.argsort(ll, kind="stable")
            uniq, starts = np.unique(ll[o2], return_index=True)
            mx = np.maximum.reduceat(aa[o2].astype(np.float32), starts) if len(starts) else []
            SPAN_HITS[uniq] += 1
            for u, m in zip(uniq.tolist(), mx):
                h = TOPS[u]
                if len(h) < 32: heapq.heappush(h, (float(m), i))
                elif m > h[0][0]: heapq.heapreplace(h, (float(m), i))
        del feats
        if (b0 // BS) % 50 == 0:
            print(f"  spans {b0}/{len(order)} · tok {total_tok} · pairs {sum(len(x) for x in PL)} · {time.time()-t0:.0f}s")
PS, PT, PL, PA = map(np.concatenate, (PS, PT, PL, PA))
print(f"done | spans {len(spans)} tok {total_tok} pairs {len(PL)} ({PL.nbytes/1e6:.0f}MB latent col)")

  spans 0/43669 · tok 288 · pairs 19049 · 1s
  spans 1200/43669 · tok 14688 · pairs 934444 · 10s
  spans 2400/43669 · tok 29088 · pairs 1850404 · 18s
  spans 3600/43669 · tok 44527 · pairs 2831253 · 27s
  spans 4800/43669 · tok 60127 · pairs 3821818 · 36s
  spans 6000/43669 · tok 76876 · pairs 4890553 · 45s
  spans 7200/43669 · tok 93676 · pairs 5953660 · 53s
  spans 8400/43669 · tok 111454 · pairs 7093787 · 62s
  spans 9600/43669 · tok 129454 · pairs 8232589 · 71s
  spans 10800/43669 · tok 148607 · pairs 9444041 · 80s
  spans 12000/43669 · tok 168049 · pairs 10692002 · 89s
  spans 13200/43669 · tok 188449 · pairs 11990259 · 99s
  spans 14400/43669 · tok 209603 · pairs 13334978 · 109s
  spans 15600/43669 · tok 231375 · pairs 14718052 · 118s
  spans 16800/43669 · tok 254175 · pairs 16158583 · 128s
  spans 18000/43669 · tok 277826 · pairs 17656260 · 138s
  spans 19200/43669 · tok 302103 · pairs 19214087 · 148s
  spans 20400/43669 · tok 327303 · pairs 20807262 · 158s
  spans 21600/43669 ·

In [10]:
# 8 · pick latents ON THE MIX (this is what lets code/CoT latents survive) — no filler regex; the model grades instead
n_spans = len(spans)
gmean = ACT_SUM / max(total_tok, 1)
cut = np.quantile(gmean[gmean > 0], 1 - DROP_TOP_MEAN)        # drop the always-on top slice
keep = (SPAN_HITS >= MIN_POS) & (SPAN_HITS <= MAX_FREQ * n_spans) & (gmean < cut)
V = np.where(keep)[0]
if len(V) > N_KEEP: V = V[np.argsort(-SPAN_HITS[V])[:N_KEEP]]
V = np.sort(V); V_SET = set(V.tolist())
dom_of = np.array([{"web":0,"code":1,"cot":2,"math":3,"chat":4}[s["dom"]] for s in spans], np.int8)
print(f"kept {len(V)} / {D_SAE} latents | span-freq median {np.median(SPAN_HITS[V]/n_spans):.3f}")
lat_order = np.argsort(PL, kind="stable")                     # latent -> its pair rows, for packet mining
lat_lo = np.searchsorted(PL[lat_order], V); lat_hi = np.searchsorted(PL[lat_order], V, side="right")

kept 2048 / 16384 latents | span-freq median 0.042


In [11]:
# 9 · packets: per latent, top firing spans (doc-deduped, cross-domain) with «witness» tokens + contrast spans
def render(si, rows_t, rows_a, n_wit=3):
    toks = list(spans[si]["toks"]); marks = rows_t[np.argsort(-rows_a)][:n_wit]
    for t in marks: toks[t] = "«" + toks[t] + "»"
    return " ".join(toks)
PACKETS = {}
rng = np.random.default_rng(RANDOM_SEED)
for k, lat in enumerate(V):
    rows = lat_order[lat_lo[k]:lat_hi[k]]
    s_of, t_of, a_of = PS[rows], PT[rows], PA[rows].astype(np.float32)
    top_spans = [si for _, si in sorted(TOPS[lat], reverse=True)]
    seen_doc, shown, fire_set = set(), [], set(np.unique(s_of).tolist())
    for si in top_spans:
        d = spans[si]["doc"]
        if d in seen_doc: continue
        seen_doc.add(d); m = s_of == si
        shown.append({"dom": spans[si]["dom"], "txt": render(si, t_of[m], a_of[m])})
        if len(shown) >= N_SHOW: break
    from itertools import islice
    cold_iter = (int(si) for si in rng.permutation(n_spans) if int(si) not in fire_set)
    cold = list(islice(cold_iter, N_CONTRAST + N_VAL // 2))
    val_pos = [int(si) for si in fire_set if si not in set(top_spans[:N_SHOW])]
    rng.shuffle(val_pos)
    PACKETS[int(lat)] = {"shown": shown, "contrast": [spans[si]["text"][:300] for si in cold[:N_CONTRAST]],
                         "val_pos": val_pos[:N_VAL // 2], "val_neg": cold[N_CONTRAST:]}
    if k % 400 == 0: print(f"  packets {k}/{len(V)}")
json.dump({str(k): {"n_shown": len(p["shown"])} for k, p in PACKETS.items()}, open(f"{OUT}/packet_stats.json", "w"))
print("packets", len(PACKETS))

  packets 0/2048
  packets 400/2048
  packets 800/2048
  packets 1200/2048
  packets 1600/2048
  packets 2000/2048
packets 2048


In [12]:
# 10 · PRICE dry-run before spending — live OpenRouter prices x actual packet sizes
import requests
_models = {m["id"]: m["pricing"] for m in requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]}
def price(mid): p = _models[mid]; return float(p["prompt"]) * 1e6, float(p["completion"]) * 1e6
def toks(s): return len(s) // 4 + 1
write_in = sum(300 + sum(toks(x["txt"]) for x in p["shown"]) + sum(toks(c) for c in p["contrast"]) for p in PACKETS.values())
val_in = sum(250 + (len(p["val_pos"]) + len(p["val_neg"])) * 60 for p in PACKETS.values())
for tag, mid, tin, tout in [("write", WRITE_MODEL, write_in, 130 * len(PACKETS)), ("check", CHECK_MODEL, val_in, 50 * len(PACKETS))]:
    pi, po = price(mid)
    print(f"  {tag} {mid}: ~{tin/1e6:.1f}M in + {tout/1e6:.2f}M out -> ${ (tin*pi + tout*po)/1e6:.2f}")
PIN_W, POUT_W = price(WRITE_MODEL); PIN_C, POUT_C = price(CHECK_MODEL)

  write deepseek/deepseek-v4-flash: ~2.9M in + 0.27M out -> $0.31
  check deepseek/deepseek-v4-flash: ~2.5M in + 0.10M out -> $0.24


In [15]:
print("key set:", bool(OPENROUTER_KEY), "| len", len(OPENROUTER_KEY))
import requests
OPENROUTER_KEY = os.environ.get("OPENROUTER_KEY","")  # set your key in the environment
r = requests.post("https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": "Bearer " + OPENROUTER_KEY},
    json={"model": WRITE_MODEL, "messages": [{"role": "user", "content": "say ok"}], "max_tokens": 5}, timeout=30)
print(r.status_code, r.text[:300])


key set: False | len 0
200 
         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         
{"i


In [18]:
# 11 · WRITE the captions (threaded, resumable, live progress) — fast non-thinking writer, provenance-tagged
assert OPENROUTER_KEY.startswith("sk-or-"), "set OPENROUTER_KEY first"
WRITE_MODEL = CHECK_MODEL = "google/gemini-2.5-flash-lite"   # fast + non-thinking; checker inherits it too
PIN_W = PIN_C = 0.10; POUT_W = POUT_C = 0.40                 # $/M for spend tracking
N_THREADS = 32
SPEND, LOCK, FAILS = [0.0], threading.Lock(), []
def ask(mid, prompt, pin, pout, max_tok=300):
    err = None
    for attempt in range(4):
        try:
            r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": "Bearer " + OPENROUTER_KEY},
                json={"model": mid, "messages": [{"role": "user", "content": prompt}],
                      "reasoning": {"enabled": False},       # no hidden thinking tokens (the deepseek slowdown)
                      "temperature": 0.2, "max_tokens": max_tok}, timeout=90)
            j = r.json(); u = j.get("usage", {})
            with LOCK: SPEND[0] += (u.get("prompt_tokens", 0) * pin + u.get("completion_tokens", 0) * pout) / 1e6
            txt = j["choices"][0]["message"]["content"]
            return json.loads(txt[txt.index("{"): txt.rindex("}") + 1])
        except Exception as e:
            err = type(e).__name__ + ": " + str(e)[:80]
            time.sleep(1 + 2 * attempt)
    with LOCK: FAILS.append(err)
    return None
W_PROMPT = (
    "You are captioning one latent of a sparse autoencoder trained on a language model.\n"
    "FIRING SPANS (strongest witness tokens marked like «this»):\n{fires}\n\n"
    "CONTRAST (the latent does NOT fire here):\n{cold}\n\n"
    "Write what this latent detects, grounded in the witness tokens. Reply ONLY with JSON:\n"
    '{{"caption": "<one specific line, max 140 chars, no filler like in-various-contexts>",'
    ' "trigger": "<the token-level pattern that fires it>", "specificity": <0-5, 5=crisp concept, 0=generic filler>}}')
done = set()
if os.path.exists(f"{OUT}/relabels.jsonl"):
    done = {json.loads(l)["latent"] for l in open(f"{OUT}/relabels.jsonl")}   # earlier writes stay untouched; we only ADD
todo = [l for l in PACKETS if l not in done]
outf = open(f"{OUT}/relabels.jsonl", "a"); of_lock = threading.Lock()
def do_write(lat):
    p = PACKETS[lat]
    fires = "\n".join(f'- [{x["dom"]}] {x["txt"][:400]}' for x in p["shown"])
    cold = "\n".join("- " + c[:200] for c in p["contrast"])
    res = ask(WRITE_MODEL, W_PROMPT.format(fires=fires, cold=cold), PIN_W, POUT_W)
    if res and res.get("caption"):
        rec = {"latent": lat, **{k: res.get(k) for k in ("caption", "trigger", "specificity")}, "writer": WRITE_MODEL}
        with of_lock: outf.write(json.dumps(rec) + "\n"); outf.flush()
        return res
    return None
print(f"todo {len(todo)} (resuming past {len(done)}) · {N_THREADS} threads · {WRITE_MODEL}")
ok, t0 = 0, time.time()
with cf.ThreadPoolExecutor(N_THREADS) as ex:
    futs = [ex.submit(do_write, l) for l in todo]
    for n, fu in enumerate(cf.as_completed(futs), 1):
        res = fu.result()
        if res:
            ok += 1
            if ok == 1: print("  first caption:", json.dumps(res)[:150])
        elif FAILS and (n - ok) <= 3: print("  FAIL:", FAILS[-1])
        if n == 20 and ok == 0: raise RuntimeError("first 20 calls all failed — fix key/model, re-run (nothing lost)")
        if n % 50 == 0 or n == len(todo):
            rate = n / (time.time() - t0)
            print(f"  {n}/{len(todo)} · ok {ok} fail {n-ok} · {rate:.1f}/s · eta {(len(todo)-n)/max(rate,.01)/60:.1f}m · ${SPEND[0]:.2f}")
outf.close()
RELABELS = {json.loads(l)["latent"]: json.loads(l) for l in open(f"{OUT}/relabels.jsonl")}
assert len(RELABELS) == len(done) + ok, "latent collision in relabels.jsonl — should be impossible, tell Claude"
print(f"captions written: {len(RELABELS)} · spend ${SPEND[0]:.2f} · fails {len(FAILS)}")


todo 1884 (resuming past 164) · 32 threads · google/gemini-2.5-flash-lite
  first caption: {"caption": "Detects phrases related to 'period of time' and 'division' or 'divisible by'.", "trigger": "period of time|divisible by", "specificity": 
  50/1884 · ok 50 fail 0 · 24.9/s · eta 1.2m · $0.01
  100/1884 · ok 100 fail 0 · 30.7/s · eta 1.0m · $0.02
  150/1884 · ok 150 fail 0 · 30.7/s · eta 0.9m · $0.03
  200/1884 · ok 200 fail 0 · 30.2/s · eta 0.9m · $0.04
  250/1884 · ok 250 fail 0 · 31.1/s · eta 0.9m · $0.04
  300/1884 · ok 300 fail 0 · 32.1/s · eta 0.8m · $0.05
  350/1884 · ok 350 fail 0 · 32.5/s · eta 0.8m · $0.06
  400/1884 · ok 400 fail 0 · 32.8/s · eta 0.8m · $0.07
  450/1884 · ok 450 fail 0 · 33.1/s · eta 0.7m · $0.08
  500/1884 · ok 500 fail 0 · 32.9/s · eta 0.7m · $0.09
  550/1884 · ok 550 fail 0 · 33.3/s · eta 0.7m · $0.10
  600/1884 · ok 600 fail 0 · 33.5/s · eta 0.6m · $0.11
  650/1884 · ok 650 fail 0 · 33.9/s · eta 0.6m · $0.12
  700/1884 · ok 700 fail 0 · 34.2/s · eta 0.6

In [19]:
# 11b · PIGEONHOLE: no two latents may share a caption — embed captions, group near-duplicates, rewrite each group CONTRASTIVELY
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
_emb = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
lats = sorted(RELABELS)
E = _emb.encode([RELABELS[l]["caption"] for l in lats], normalize_embeddings=True, batch_size=256)
sim = E @ E.T; np.fill_diagonal(sim, 0)
parent = list(range(len(lats)))                               # union-find over caption-collision pairs
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in zip(*np.where(np.triu(sim, 1) > 0.88)): parent[find(int(a))] = find(int(b))
groups = {}
for i in range(len(lats)): groups.setdefault(find(i), []).append(lats[i])
groups = [g for g in groups.values() if len(g) > 1]
print(f"collision groups: {len(groups)} covering {sum(len(g) for g in groups)} latents "
      f"(largest {max((len(g) for g in groups), default=0)})")

P_PROMPT = (
    "These {n} sparse-autoencoder latents were all given nearly the SAME caption, but they are DIFFERENT latents.\n"
    "For each, here are its firing spans (witness tokens marked «like this»):\n\n{blocks}\n"
    "Write a DISTINCT caption per latent that captures what separates it from the others in this group, "
    "grounded in the witnesses. Reply ONLY with JSON: {{\"captions\": {{\"<latent_id>\": \"<distinct caption, max 140 chars>\", ...}}}}")
def do_group(g):
    g = g[:6]                                                  # cap the joint packet; >6-way collisions rewrite in chunks next run
    blocks = "\n".join("LATENT " + str(l) + " (current: " + RELABELS[l]["caption"][:100] + "):\n"
                        + "\n".join("- [" + x["dom"] + "] " + x["txt"][:300] for x in PACKETS[l]["shown"][:8])
                        for l in g)
    res = ask(WRITE_MODEL, P_PROMPT.format(n=len(g), blocks=blocks), PIN_W, POUT_W, max_tok=200 + 60 * len(g))
    if not res: return 0
    hit = 0
    for l in g:
        c = (res.get("captions") or {}).get(str(l))
        if isinstance(c, str) and len(c) > 10: RELABELS[l]["caption"] = c; RELABELS[l]["pigeonholed"] = True; hit += 1
    return hit
fixed = 0
with cf.ThreadPoolExecutor(N_THREADS) as ex:
    for n_done in ex.map(do_group, groups): fixed += n_done
with open(f"{OUT}/relabels.jsonl", "w") as f:                  # rewrite checkpoint with disambiguated captions
    for l in sorted(RELABELS): f.write(json.dumps({"latent": l, **RELABELS[l]}) + "\n")
print(f"rewrote {fixed} captions across {len(groups)} groups · spend ${SPEND[0]:.2f} · validation (next cell) now checks the FINAL captions")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

collision groups: 69 covering 1138 latents (largest 932)
rewrote 181 captions across 69 groups · spend $0.43 · validation (next cell) now checks the FINAL captions


In [20]:
# 12 · VALIDATE: caption must PREDICT which held-out spans fire (auto-interp scoring) -> quality gate
C_PROMPT = (
    "A latent in a sparse autoencoder is described as: \"{cap}\"\n"
    "For each numbered span below, decide if this latent fires in it.\n{quiz}\n"
    'Reply ONLY with JSON: {{"fires": [<numbers>]}}')
outf = open(f"{OUT}/validated.jsonl", "a"); of_lock = threading.Lock()
done_v = {json.loads(l)["latent"] for l in open(f"{OUT}/validated.jsonl")} if os.path.getsize(f"{OUT}/validated.jsonl") else set()
def do_check(lat):
    p, rl = PACKETS[lat], RELABELS.get(lat)
    if not rl or not p["val_pos"] or not p["val_neg"]: return
    items = [(si, 1) for si in p["val_pos"]] + [(si, 0) for si in p["val_neg"]]
    random.Random(lat).shuffle(items)
    quiz = "\n".join(f"{n}. {spans[si]['text'][:240]}" for n, (si, _) in enumerate(items))
    res = ask(CHECK_MODEL, C_PROMPT.format(cap=rl["caption"], quiz=quiz), PIN_C, POUT_C, max_tok=120)
    if res is None: return
    pred = set(int(x) for x in res.get("fires", []) if isinstance(x, (int, float)))
    truth = {n for n, (_, y) in enumerate(items) if y}
    tp = len(pred & truth); f1 = 2 * tp / max(len(pred) + len(truth), 1)
    with of_lock: outf.write(json.dumps({"latent": lat, "f1": round(f1, 3)}) + "\n"); outf.flush()
with cf.ThreadPoolExecutor(N_THREADS) as ex:
    todo = [l for l in RELABELS if l not in done_v]
    for n, _ in enumerate(ex.map(do_check, todo)):
        if n % 100 == 0: print(f"  checked {n}/{len(todo)} · spend ${SPEND[0]:.2f}")
outf.close()
F1S = {json.loads(l)["latent"]: json.loads(l)["f1"] for l in open(f"{OUT}/validated.jsonl")}
FINAL = {l: r for l, r in RELABELS.items()
         if (r.get("specificity") or 0) >= KEEP_SPECIFICITY and F1S.get(l, 0) >= KEEP_VAL_F1}
json.dump({str(l): r["caption"] for l, r in FINAL.items()}, open(f"{OUT}/labels_final.json", "w"), indent=1)
print(f"validated {len(F1S)} · PASSED GATE {len(FINAL)} / {len(RELABELS)} "
      f"(specificity>={KEEP_SPECIFICITY} & F1>={KEEP_VAL_F1}) · spend ${SPEND[0]:.2f}")

  checked 0/2045 · spend $0.48
  checked 100/2045 · spend $0.50
  checked 200/2045 · spend $0.57
  checked 300/2045 · spend $0.57
  checked 400/2045 · spend $0.59
  checked 500/2045 · spend $0.60
  checked 600/2045 · spend $0.60
  checked 700/2045 · spend $0.60
  checked 800/2045 · spend $0.60
  checked 900/2045 · spend $0.60
  checked 1000/2045 · spend $0.63
  checked 1100/2045 · spend $0.63
  checked 1200/2045 · spend $0.63
  checked 1300/2045 · spend $0.65
  checked 1400/2045 · spend $0.65
  checked 1500/2045 · spend $0.65
  checked 1600/2045 · spend $0.65
  checked 1700/2045 · spend $0.68
  checked 1800/2045 · spend $0.71
  checked 1900/2045 · spend $0.71
  checked 2000/2045 · spend $0.71


KeyError: 'f1'

In [21]:
# 12b · tolerant tally of the validation results (no re-spend — reads what's on disk)
F1S, bad = {}, 0
for l in open(f"{OUT}/validated.jsonl"):
    try:
        d = json.loads(l)
        if "f1" in d: F1S[d["latent"]] = d["f1"]
        else: bad += 1
    except json.JSONDecodeError: bad += 1
FINAL = {l: r for l, r in RELABELS.items()
         if (r.get("specificity") or 0) >= KEEP_SPECIFICITY and F1S.get(l, 0) >= KEEP_VAL_F1}
json.dump({str(l): r["caption"] for l, r in FINAL.items()}, open(f"{OUT}/labels_final.json", "w"), indent=1)
import numpy as np
f1s = np.array(list(F1S.values()))
specs = np.array([RELABELS[l].get("specificity") or 0 for l in RELABELS])
print(f"validated {len(F1S)} (skipped {bad} malformed) · PASSED GATE {len(FINAL)} / {len(RELABELS)}")
print(f"F1: mean {f1s.mean():.3f} · >=0.6 {(f1s >= 0.6).mean():.1%} | specificity: mean {specs.mean():.2f} · >=2 {(specs >= 2).mean():.1%}")
for l in list(FINAL)[:5]: print(f"  [{l}] {FINAL[l]['caption'][:110]}")


validated 2038 (skipped 513 malformed) · PASSED GATE 874 / 2045
F1: mean 0.534 · >=0.6 43.2% | specificity: mean 3.54 · >=2 99.5%
  [100] The concept of a single countable entity or unit.
  [44] Firing on lists of entities or actions leading to a result, often with prepositions or conjunctions.
  [12] Negation marker, often contrasting a prior statement or expectation
  [79] Large round numbers (thousands, tens of thousands, millions) and quantifiers like 'thousands of'
  [120] Detects 'the' followed by 'first', 'latest', 'next', 'only', or 'last'


In [22]:
# 13 · new 32k BPE tokenizer trained on THIS corpus (the mega golf model reads these ids, not Gemma's)
import sentencepiece as spm
with open("/content/corpus.txt", "w") as f:
    for dom, text in docs: f.write(text + "\n")
spm.SentencePieceTrainer.train(input="/content/corpus.txt", model_prefix=f"{OUT}/mega_tok",
    vocab_size=TOK_VOCAB, model_type="bpe", character_coverage=0.9995, byte_fallback=True,
    input_sentence_size=2000000, shuffle_input_sentence=True)
sp = spm.SentencePieceProcessor(model_file=f"{OUT}/mega_tok.model")
print("tokenizer ready | vocab", sp.vocab_size(), "| sample:", sp.encode("def merge_intervals(intervals):", out_type=str)[:8])

tokenizer ready | vocab 32768 | sample: ['▁def', '▁merge', '_', 'interval', 's', '(', 'interval', 's']


In [23]:
# 14 · EXPORT the bundle: labels + training set (spans + per-token firings in V-space) + tokenizer -> zip
import shutil
lut = np.full(D_SAE, -1, np.int32); lut[V] = np.arange(len(V), dtype=np.int32)
mask = lut[PL] >= 0
tl = lut[PL[mask]]                                            # targets remapped to V-space
np.savez_compressed(f"{OUT}/targets.npz", span=PS[mask], tok=PT[mask], vlat=tl, act=PA[mask],
                    V=V.astype(np.int32), n_spans=n_spans)
with gzip.open(f"{OUT}/spans.jsonl.gz", "wt") as f:            # teacher token strings kept -> re-align targets to ANY tokenizer by char overlap
    for s in spans: f.write(json.dumps({"i": s["i"], "doc": s["doc"], "dom": s["dom"], "text": s["text"], "toks": s["toks"]}) + "\n")
json.dump({"teacher": {"model": MODEL_NAME, "sae": SAE_ID}, "corpus": dict(Counter(s["dom"] for s in spans)),
           "spans": n_spans, "tokens": int(total_tok), "kept_latents": len(V),
           "captions_written": len(RELABELS), "passed_gate": len(FINAL),
           "writer": WRITE_MODEL, "checker": CHECK_MODEL, "spend_usd": round(SPEND[0], 2),
           "gates": {"specificity": KEEP_SPECIFICITY, "val_f1": KEEP_VAL_F1}},
          open(f"{OUT}/results.json", "w"), indent=1)
shutil.make_archive("/content/relabel_bundle", "zip", OUT)
print("bundle:", sorted(os.listdir(OUT)), f"| zip {os.path.getsize('/content/relabel_bundle.zip')/1e6:.1f} MB")
from google.colab import files; files.download("/content/relabel_bundle.zip")

bundle: ['labels_final.json', 'mega_tok.model', 'mega_tok.vocab', 'packet_stats.json', 'relabels.jsonl', 'results.json', 'spans.jsonl.gz', 'targets.npz', 'validated.jsonl'] | zip 30.1 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>